# LLaMA + RAG for ITS-style questions (Web URL demo)

This notebook demonstrates **Retrieval-Augmented Generation (RAG)** using:
- an open-source LLM served locally via **Ollama** (e.g., LLaMA 3 or LLaMA 2), and
- a lightweight vector database (**Chroma**) for retrieval.

RAG is the key idea behind *grounded* answers:
> Instead of relying only on what the model memorized during training, we retrieve relevant source text and include it in the prompt.

---

## What you will build

1. Load a web page (URL)
2. Split it into chunks
3. Embed chunks into vectors and store them in Chroma
4. Retrieve the top-k relevant chunks for a user question
5. Ask the LLM to answer **using only the retrieved context**
6. (Optional) Launch a small **Gradio** UI

---

## Inputs
- A URL (default example uses a Wikipedia page)

## Outputs
- Console answers to test questions
- (Optional) A running Gradio app


## 0) Install dependencies (Colab)

This notebook is designed for **Google Colab**.

If you are running locally, you can:
- install Ollama on your machine, and
- run `ollama serve` in a separate terminal.

In Colab, we use `colab-xterm` so you can run Ollama in a terminal tab.


In [ ]:
!pip -q install colab-xterm
%load_ext colabxterm

### Start Ollama server (in the xterm)

Run this cell, then in the terminal that opens run:

```bash
ollama serve
```

In a **second** xterm tab (or after it is running), pull models:

```bash
ollama pull llama3
ollama pull nomic-embed-text
```

If llama3 is too large for your runtime, you can use:

```bash
ollama pull llama2
```


In [ ]:
%xterm

## 1) Configure models

- `LLM_MODEL` is the chat model (LLaMA 3 or LLaMA 2)
- `EMBED_MODEL` converts text into vectors (we use `nomic-embed-text`)


In [ ]:
LLM_MODEL = "llama3"      # or "llama2"
EMBED_MODEL = "nomic-embed-text"


## 2) Quick smoke test: can we call the LLM?

This checks your Ollama server is reachable from Python.


In [ ]:
!pip -q install ollama

In [ ]:
import ollama

resp = ollama.chat(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence and tell me you are ready for RAG."}]
)
print(resp["message"]["content"])


## 3) Load documents from a Web URL

For the workshop we use a web page as the “knowledge base”.
In a transport setting this could be:
- a traffic bulletin page
- a safety report
- operational procedures
- a maintenance manual

Here we demonstrate with a public Wikipedia page.


In [ ]:
!pip -q install langchain chromadb beautifulsoup4 gradio

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

URL = "https://en.wikipedia.org/wiki/Ohiya"
loader = WebBaseLoader(URL)
docs = loader.load()

print("Loaded documents:", len(docs))
print("Example characters:", len(docs[0].page_content))


## 4) Chunking + embeddings + vector store

Why chunking?
- LLMs have context limits.
- Retrieval works better with smaller chunks.

We split the document text into overlapping chunks, embed them, and store them in Chroma.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(docs)
print("Chunks:", len(chunks))

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# Create an in-memory Chroma DB (persist_directory is optional)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_web_demo"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


## 5) Define the RAG answer function

RAG pattern:

1. Retrieve top-k chunks relevant to the question
2. Build a prompt: **Context + Question**
3. Ask the LLM to answer using the context

We also instruct the model:
- if the answer is not in the context, say “I don’t know”.


In [ ]:
def rag_answer(question: str) -> str:
    retrieved = retriever.get_relevant_documents(question)
    context = "\n\n".join([d.page_content for d in retrieved])

    prompt = f"""You are a helpful assistant.
Use ONLY the context below to answer the question.
If the answer is not contained in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp["message"]["content"]


## 6) Test RAG in the notebook

In [ ]:
print(rag_answer("What is Ohiya? Give a short answer."))

## 7) Optional: Launch a Gradio UI

This creates a simple textbox interface so workshop participants can try multiple questions quickly.


In [ ]:
import gradio as gr

iface = gr.Interface(
    fn=rag_answer,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the loaded URL..."),
    outputs="text",
    title="RAG with Ollama (LLaMA) + Chroma",
    description=f"URL loaded: {URL}"
)

iface.launch(debug=False)
